# ERA5 다운로드 — North Sea White-fronted Geese

## 다운로드 설정
| 항목 | 값 |
|------|----|
| 데이터셋 | ERA5 hourly on pressure levels |
| 변수 | u, v, w, t, z, q, r, cc (8개) |
| 기압면 | 850 / 925 / 1000 hPa |
| 연도 | 2014, 2015, 2016, 2017 |
| 기간 | 9~11월 |
| 시간 | 00:00 ~ 23:00 |
| 영역 | N78 / W3 / S44 / E116 |
| 포맷 | NetCDF |

## 파일 구조
용량 초과 방지를 위해 월별 분할 저장. 실패 시 주 단위 자동 분할.
```
BirdXAI/era5_geese/
  era5_geese_2014_09.nc
  era5_geese_2014_09_w1.nc  (월 단위 실패 시)
  ...
```

## 실행 순서
1. Cell 1 — 설치 및 세션 유지
2. Cell 2 — Google Drive 연결
3. Cell 3 — CDS API 키 입력 및 함수 정의
4. Cell 4~7 — 연도별 다운로드
5. Cell 8 — 전체 현황 확인

> CDS API 키 발급: https://cds.climate.copernicus.eu/how-to-api

In [ ]:
# ── Cell 1: 설치 및 세션 유지 ─────────────────────────────────────────
!pip install cdsapi -q
print('설치 완료!')

from IPython.display import Javascript
display(Javascript('''
function KeepAlive() {
  document.querySelector('#top-toolbar').click();
  console.log('Session alive:', new Date());
  setTimeout(KeepAlive, 60000);
}
KeepAlive();
console.log('세션 유지 시작 (60초마다)');
'''))

설치 완료!


<IPython.core.display.Javascript object>

In [ ]:
# ── Cell 2: Google Drive 연결 ─────────────────────────────────────────
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

OUTPUT_DIR = Path('/content/drive/MyDrive/data/ERA5_download_geese')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'저장 폴더: {OUTPUT_DIR}')
existing = sorted(OUTPUT_DIR.glob('*.nc'))
print(f'기존 파일 수: {len(existing)}개')
for f in existing:
    print(f'  {f.name}  ({f.stat().st_size/1024**2:.0f} MB)')

Mounted at /content/drive
저장 폴더: /content/drive/MyDrive/data/ERA5_download_geese
기존 파일 수: 0개


In [ ]:
# ── Cell 3: CDS API 설정 및 함수 정의 ────────────────────────────────
import os
import calendar
import cdsapi

# ▼ 여기에 CDS API 키 입력
CDS_API_KEY = 'key값 입력'

os.environ['CDSAPI_URL'] = 'https://cds.climate.copernicus.eu/api'
os.environ['CDSAPI_KEY'] = CDS_API_KEY

# 다운로드 설정
VARIABLES = [
    'u_component_of_wind',       # u: 동서 풍속
    'v_component_of_wind',       # v: 남북 풍속
    'vertical_velocity',         # w: 수직기류
    'temperature',               # t: 기온
    'geopotential',              # z: 지오퍼텐셜
    'specific_humidity',         # q: 비습
    'relative_humidity',         # r: 상대습도
    'fraction_of_cloud_cover',   # cc: 구름량
]
LEVELS   = ['1000', '925', '850']
HOURS    = [f'{h:02d}:00' for h in range(24)]
AREA     = [78, -3, 44, 116]  # N78 / W3 / S44 / E116

# 연도별 개체 정보 (참고용)
YEAR_INFO = {
    2014: '701, 707, 709, 716, 720, 742, 749, 750, 768, Frank_3084 등',
    2015: '711, 712, 738, 766, Hannah_3988, Jouri_3992 등',
    2016: 'GWFG_2015_408~450 계열',
    2017: 'GWFG_2015_410~450 계열, KOL_02~42, 52_EvertII 등',
}


def _week_ranges(n_days):
    """월을 4개 주 단위로 분할"""
    return [
        list(range(1,  8)),
        list(range(8,  15)),
        list(range(15, 22)),
        list(range(22, n_days + 1)),
    ]


def _request(year, month, days, output_path):
    """CDS API 요청 (내부용)"""
    client = cdsapi.Client()
    client.retrieve(
        'reanalysis-era5-pressure-levels',
        {
            'product_type': ['reanalysis'],
            'variable':       VARIABLES,
            'pressure_level': LEVELS,
            'year':  [str(year)],
            'month': [f'{month:02d}'],
            'day':   [f'{d:02d}' for d in days],
            'time':  HOURS,
            'area':  AREA,
            'data_format': 'netcdf',
        },
        str(output_path),
    )


def download_month(year, month):
    """
    월 단위 다운로드 시도.
    cost limits exceeded 오류 시 자동으로 주 단위(4개)로 분할 재시도.
    이미 완료된 파일은 스킵.
    """
    n_days     = calendar.monthrange(year, month)[1]
    month_path = OUTPUT_DIR / f'era5_geese_{year}_{month:02d}.nc'
    week_paths = [
        OUTPUT_DIR / f'era5_geese_{year}_{month:02d}_w{i+1}.nc'
        for i in range(4)
    ]

    # 이미 완료된 경우 스킵
    if month_path.exists():
        mb = month_path.stat().st_size / 1024**2
        print(f'  [SKIP] {month_path.name}  ({mb:.0f} MB)')
        return
    if all(p.exists() for p in week_paths):
        print(f'  [SKIP] {year}-{month:02d} 주 단위 파일 모두 완료')
        return

    # 1차 시도: 월 단위
    print(f'  [{year}-{month:02d}] 월 단위 요청 중... ({n_days}일 × 24시간)')
    try:
        _request(year, month, list(range(1, n_days + 1)), month_path)
        mb = month_path.stat().st_size / 1024**2
        print(f'  [완료] {month_path.name}  →  {mb:.0f} MB')
        return
    except Exception as e:
        if month_path.exists():
            month_path.unlink()
        if 'cost limits exceeded' not in str(e):
            raise
        print(f'  [용량 초과] 주 단위로 분할 재시도...')

    # 2차 시도: 주 단위 (4개)
    for i, days in enumerate(_week_ranges(n_days)):
        week_path = week_paths[i]
        if week_path.exists():
            mb = week_path.stat().st_size / 1024**2
            print(f'  [SKIP] {week_path.name}  ({mb:.0f} MB)')
            continue
        print(f'  [{year}-{month:02d} W{i+1}] {days[0]}~{days[-1]}일 ({len(days)}일)')
        try:
            _request(year, month, days, week_path)
            mb = week_path.stat().st_size / 1024**2
            print(f'  [완료] {week_path.name}  →  {mb:.0f} MB')
        except Exception as e:
            if week_path.exists():
                week_path.unlink()
            print(f'  [ERROR] {week_path.name} 실패: {e}')
            raise


def download_year(year):
    """9~11월 순차 다운로드"""
    print(f'\n========== {year}년 ({YEAR_INFO[year]}) ==========')
    for month in [9, 10, 11]:
        download_month(year, month)
    print(f'========== {year}년 완료 ==========')


def check_status():
    """전체 다운로드 현황 출력"""
    print(f'\n{"파일명":<40} {"크기":>10}')
    print('-' * 52)
    total_mb = 0
    for year in [2014, 2015, 2016, 2017]:
        for month in [9, 10, 11]:
            p = OUTPUT_DIR / f'era5_geese_{year}_{month:02d}.nc'
            if p.exists():
                mb = p.stat().st_size / 1024**2
                total_mb += mb
                print(f'{p.name:<40} {mb:>8.0f} MB')
                continue
            week_info = []
            for i in range(4):
                wp = OUTPUT_DIR / f'era5_geese_{year}_{month:02d}_w{i+1}.nc'
                if wp.exists():
                    mb = wp.stat().st_size / 1024**2
                    total_mb += mb
                    week_info.append(f'W{i+1}({mb:.0f}MB)')
            if week_info:
                print(f'era5_geese_{year}_{month:02d} (주단위): {" ".join(week_info)}')
            else:
                print(f'{"era5_geese_"+str(year)+"_"+f"{month:02d}"+".nc":<40} {"미완료":>10}')
    print('-' * 52)
    print(f'총 {total_mb:.0f} MB  ({total_mb/1024:.1f} GB)')


print('설정 완료! Cell 4부터 연도별로 실행하세요.')
print(f'다운로드 영역: N{AREA[0]} / W{abs(AREA[1])} / S{AREA[2]} / E{AREA[3]}')
print(f'변수: {len(VARIABLES)}개 × 기압면 {len(LEVELS)}개 (1000/925/850hPa)')

설정 완료! Cell 4부터 연도별로 실행하세요.
다운로드 영역: N78 / W3 / S44 / E116
변수: 8개 × 기압면 3개 (1000/925/850hPa)


In [ ]:
# ── Cell 4: 2014년 다운로드 ───────────────────────────────────────────
download_year(2014)


========== 2014년 (701, 707, 709, 716, 720, 742, 749, 750, 768, Frank_3084 등) ==========
  [2014-09] 월 단위 요청 중... (30일 × 24시간)
  [용량 초과] 주 단위로 분할 재시도...
  [2014-09 W1] 1~7일 (7일)


2026-05-30 19:32:50,669 INFO Request ID is 08b71989-c169-4ce3-89e4-7c0b2732a62c
INFO:ecmwf.datastores.legacy_client:Request ID is 08b71989-c169-4ce3-89e4-7c0b2732a62c
2026-05-30 19:32:51,956 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-05-30 19:33:43,224 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-05-30 19:37:13,074 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


14c70267e3f4ba65ab4da5bad988e513.nc:   0%|          | 0.00/400M [00:00<?, ?B/s]

  [완료] era5_geese_2014_09_w1.nc  →  400 MB
  [2014-09 W2] 8~14일 (7일)


2026-05-30 19:38:52,828 INFO Request ID is db103c15-99d0-42bc-b6d4-2417eedd5a50
INFO:ecmwf.datastores.legacy_client:Request ID is db103c15-99d0-42bc-b6d4-2417eedd5a50
2026-05-30 19:38:53,047 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-05-30 19:39:26,220 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-05-30 19:39:43,431 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-05-30 19:40:09,199 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-05-30 19:43:13,048 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


...

4088491376b57c2cf8e4eaf3989605b7.nc:   0%|          | 0.00/409M [00:00<?, ?B/s]

  [완료] era5_geese_2014_11_w3.nc  →  409 MB
  [2014-11 W4] 22~30일 (9일)


2026-05-30 20:33:07,703 INFO Request ID is 4e188237-1cdc-4e61-9114-ac0001194fc6
INFO:ecmwf.datastores.legacy_client:Request ID is 4e188237-1cdc-4e61-9114-ac0001194fc6
2026-05-30 20:33:07,839 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-05-30 20:33:30,511 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-05-30 20:39:31,334 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


eeabe4b26fb8760649c5f459bdab4dff.nc:   0%|          | 0.00/528M [00:00<?, ?B/s]

  [완료] era5_geese_2014_11_w4.nc  →  528 MB
========== 2014년 완료 ==========


In [ ]:
# ── Cell 5: 2015년 다운로드 ───────────────────────────────────────────
download_year(2015)


========== 2015년 (711, 712, 738, 766, Hannah_3988, Jouri_3992 등) ==========
  [2015-09] 월 단위 요청 중... (30일 × 24시간)
  [용량 초과] 주 단위로 분할 재시도...
  [2015-09 W1] 1~7일 (7일)


2026-05-30 20:40:11,860 INFO Request ID is 0fc9c074-490f-4ed0-a5ad-4b2e690e2528
INFO:ecmwf.datastores.legacy_client:Request ID is 0fc9c074-490f-4ed0-a5ad-4b2e690e2528
2026-05-30 20:40:15,538 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-05-30 20:40:29,422 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-05-30 20:44:39,993 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


15d7f0d3c422eda208d632520d948cc6.nc:   0%|          | 0.00/404M [00:00<?, ?B/s]

  [완료] era5_geese_2015_09_w1.nc  →  404 MB
  [2015-09 W2] 8~14일 (7일)


2026-05-30 20:44:59,003 INFO Request ID is ede61a1d-dc3a-4979-a78b-5f35478eebc4
INFO:ecmwf.datastores.legacy_client:Request ID is ede61a1d-dc3a-4979-a78b-5f35478eebc4
2026-05-30 20:44:59,181 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-05-30 20:45:13,330 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-05-30 20:49:21,711 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


...

48d377650405c5f2f82c55e3df8f65f1.nc:   0%|          | 0.00/410M [00:00<?, ?B/s]

  [완료] era5_geese_2015_11_w3.nc  →  410 MB
  [2015-11 W4] 22~30일 (9일)


2026-05-30 21:36:10,419 INFO Request ID is c7b69d0a-786b-44b1-9bc1-058376ff4696
INFO:ecmwf.datastores.legacy_client:Request ID is c7b69d0a-786b-44b1-9bc1-058376ff4696
2026-05-30 21:36:11,642 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-05-30 21:36:27,811 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-05-30 21:40:34,254 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


8ce8ab85f41ef8edd027a94c477ae0ec.nc:   0%|          | 0.00/525M [00:00<?, ?B/s]

  [완료] era5_geese_2015_11_w4.nc  →  525 MB
========== 2015년 완료 ==========


In [ ]:
# ── Cell 6: 2016년 다운로드 ───────────────────────────────────────────
download_year(2016)


========== 2016년 (GWFG_2015_408~450 계열) ==========
  [2016-09] 월 단위 요청 중... (30일 × 24시간)
  [용량 초과] 주 단위로 분할 재시도...
  [2016-09 W1] 1~7일 (7일)


2026-05-30 21:41:01,999 INFO Request ID is 1f033b6a-96d4-49be-9937-1df44b3e3623
INFO:ecmwf.datastores.legacy_client:Request ID is 1f033b6a-96d4-49be-9937-1df44b3e3623
2026-05-30 21:41:02,120 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-05-30 21:41:23,736 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-05-30 21:45:22,047 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


28211d7dcef82cae20332f6b038a9970.nc:   0%|          | 0.00/401M [00:00<?, ?B/s]

  [완료] era5_geese_2016_09_w1.nc  →  401 MB
  [2016-09 W2] 8~14일 (7일)


2026-05-30 21:45:46,174 INFO Request ID is 181a04ca-9f90-447e-9940-97add28ff90f
INFO:ecmwf.datastores.legacy_client:Request ID is 181a04ca-9f90-447e-9940-97add28ff90f
2026-05-30 21:45:46,308 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-05-30 21:46:00,161 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-05-30 21:50:06,147 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


...

c2d207a1acbcd10220aa0de9f1ddf4e1.nc:   0%|          | 0.00/408M [00:00<?, ?B/s]

  [완료] era5_geese_2016_11_w3.nc  →  408 MB
  [2016-11 W4] 22~30일 (9일)


2026-05-30 22:51:20,873 INFO Request ID is ee28dfed-babd-4eed-a5bf-1ecf32da7a73
INFO:ecmwf.datastores.legacy_client:Request ID is ee28dfed-babd-4eed-a5bf-1ecf32da7a73
2026-05-30 22:51:21,010 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-05-30 22:51:35,284 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-05-30 22:57:46,816 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


dae693583b5a967bf174b7b5ab1513fc.nc:   0%|          | 0.00/529M [00:00<?, ?B/s]

  [완료] era5_geese_2016_11_w4.nc  →  529 MB
========== 2016년 완료 ==========


In [ ]:
# ── Cell 7: 2017년 다운로드 ───────────────────────────────────────────
download_year(2017)


========== 2017년 (GWFG_2015_410~450 계열, KOL_02~42, 52_EvertII 등) ==========
  [2017-09] 월 단위 요청 중... (30일 × 24시간)
  [용량 초과] 주 단위로 분할 재시도...
  [2017-09 W1] 1~7일 (7일)


2026-05-30 22:58:11,156 INFO Request ID is d2627e8f-b40e-48b2-81ac-3dd05cbab6a3
INFO:ecmwf.datastores.legacy_client:Request ID is d2627e8f-b40e-48b2-81ac-3dd05cbab6a3
2026-05-30 22:58:11,291 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-05-30 22:58:32,884 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-05-30 23:04:32,293 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


72402bb522b1c99440db9e1fbc16b577.nc:   0%|          | 0.00/400M [00:00<?, ?B/s]

  [완료] era5_geese_2017_09_w1.nc  →  400 MB
  [2017-09 W2] 8~14일 (7일)


2026-05-30 23:04:51,113 INFO Request ID is 09c3979d-4835-4a5f-bb36-239f935107cf
INFO:ecmwf.datastores.legacy_client:Request ID is 09c3979d-4835-4a5f-bb36-239f935107cf
2026-05-30 23:04:51,253 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-05-30 23:05:05,119 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-05-30 23:09:13,335 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


...

6ee9bab227d9c331b2c926ace763d57d.nc:   0%|          | 0.00/407M [00:00<?, ?B/s]

  [완료] era5_geese_2017_11_w3.nc  →  407 MB
  [2017-11 W4] 22~30일 (9일)


2026-05-31 00:10:44,996 INFO Request ID is df384aaf-bdcb-4256-b717-1e0ff20b6a61
INFO:ecmwf.datastores.legacy_client:Request ID is df384aaf-bdcb-4256-b717-1e0ff20b6a61
2026-05-31 00:10:45,134 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-05-31 00:11:06,720 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-05-31 00:17:05,671 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


7abb19db8ee1c4563f39d49f0e0f507e.nc:   0%|          | 0.00/530M [00:00<?, ?B/s]

  [완료] era5_geese_2017_11_w4.nc  →  530 MB
========== 2017년 완료 ==========


In [ ]:
# ── Cell 8: 전체 현황 확인 ────────────────────────────────────────────
check_status()

In [ ]:
# ── Cell 9: 파일 내용 검증 (선택) ────────────────────────────────────
import xarray as xr
import pandas as pd

files = sorted(OUTPUT_DIR.glob('*.nc'))
print(f'총 {len(files)}개 파일')
print(f'{"파일명":<40} {"시간 수":>7} {"변수":>30}')
print('-' * 80)
for f in files:
    try:
        ds = xr.open_dataset(f)
        times = pd.to_datetime(ds['valid_time'].values)
        vars_ = list(ds.data_vars)
        print(f'{f.name:<40} {len(times):>7}  {str(vars_)}')
        ds.close()
    except Exception as e:
        print(f'{f.name:<40} 읽기 오류: {e}')